In [ ]:
import torch
import random
import numpy as np
from sentence_transformers.losses import CosineSimilarityLoss,SoftmaxLoss
from setfit import SetFitModel, TrainingArguments
from setfit import Trainer
from setfit import SetFitTrainer
from sklearn.metrics import classification_report
from datasets import Dataset
import pandas as pd
import re
from numpy import mean
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.metrics import precision_score, recall_score,f1_score, matthews_corrcoef
from sklearn.metrics import roc_curve,auc,roc_auc_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
import os

In [ ]:
import wandb
wandb.init(mode="disabled")
import warnings
warnings.filterwarnings("ignore")

In [ ]:
seed = 42
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed_all(seed)

In [ ]:
import pandas as pd
folder = "data_for_use"
test_df = pd.read_csv(os.path.join(folder, "Test_Wicket.csv"))
train_df = pd.read_csv(os.path.join(folder, "Train_Wicket.csv"))

combined = pd.concat([test_df, train_df], ignore_index=True)
combined = combined.sample(frac=1, random_state=seed).reset_index(drop=True)

df = combined[["combined_text", "security"]].copy()
df = df.rename(columns={"combined_text": "text", "security": "label"})

In [ ]:
df.head(2)

In [ ]:
def preprocess(issues):
    processed_issues = []

    for issue in issues:
        issue = re.sub(r'```.*?```', ' ', issue, flags=re.DOTALL)
        issue = re.sub(r'\n', ' ', issue)
        issue = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', ' ', issue)
        issue = re.sub(r'\d+', ' ', issue)
        issue = re.sub(r'[^a-zA-Z0-9?\s]', ' ', issue)
        issue = re.sub(r'\s+', ' ', issue)
        processed_issues.append(issue)

    return processed_issues

In [ ]:
df['text'] =  preprocess(df['text'])

In [ ]:
y = df['label'].values

In [ ]:
df['label'].value_counts()

In [ ]:
# Perform 5-fold cross-validation
num_folds = 5
kf = StratifiedKFold(n_splits=num_folds)
bug_recall_scores = []
bug_precision_scores = []
bug_f1_scores = []

nonbug_recall_scores = []
nonbug_precision_scores = []
nonbug_f1_scores = []

accuracy_scores = []
auc_scores = []
mcc_scores = []
foldi = 1
for train_index, test_index in kf.split(df,y):
    print('Fold:',foldi)
    foldi = foldi+1
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model = SetFitModel.from_pretrained("sentence-transformers/all-mpnet-base-v2").to(device)
    
    train_df = df.iloc[train_index].reset_index(drop=True)
    test_df = df.iloc[test_index].reset_index(drop=True)
    
    train_set = Dataset.from_pandas(train_df)
    test_set = Dataset.from_pandas(test_df)
    
    trainer = SetFitTrainer(model=model, train_dataset=train_set, loss_class=CosineSimilarityLoss, num_iterations=10, num_epochs=1,batch_size=4)
    trainer.train()

    y_pred = trainer.model.predict(test_set['text'])

    bug_recall_scores.append(recall_score(test_set['label'], y_pred, pos_label=1))
    bug_precision_scores.append(precision_score(test_set['label'], y_pred, pos_label=1))
    bug_f1_scores.append(f1_score(test_set['label'], y_pred, pos_label=1))

    nonbug_recall_scores.append(recall_score(test_set['label'], y_pred, pos_label=0))
    nonbug_precision_scores.append(precision_score(test_set['label'], y_pred, pos_label=0))
    nonbug_f1_scores.append(f1_score(test_set['label'], y_pred, pos_label=0))

    accuracy_scores.append(accuracy_score(test_set['label'], y_pred))

#Compute average scores
print('accuracy:', mean(accuracy_scores))

print('sec_recall_scores:', mean(bug_recall_scores))
print('sec_precision_scores:', mean(bug_precision_scores))
print('sec_f1_scores:', mean(bug_f1_scores))

print('nonsec_recall_scores:', mean(nonbug_recall_scores))
print('nonsec_precision_scores:', mean(nonbug_precision_scores))
print('nonsec_f1_scores:', mean(nonbug_f1_scores))